- fastapi는 무엇이고, 언제 사용을 하는지등 기본적인 이론도 같이 공부하면 좋을 것 같습니다.
  - FastAPI , uvicorn , ...

### 01. 필요한 라이브러리 설치

In [ ]:
!pip install FastAPI -q
!pip install uvicorn -q
!pip install torch
!pip install python-multipart
!pip install fastapi nest-asyncio pyngrok uvicorn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.5/93.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 1.8 MB/s eta 0:00:00


### 02. FastAPI 실행

- 학습때 설치했던 yolov5 폴더가 있는 경로로 이동해주세요.

In [ ]:
cd /content/drive/MyDrive/comento/1

/content/drive/MyDrive/comento/1


- 학습한 모델을 로드해줍니다.

In [ ]:
import os
import io
import uvicorn
import nest_asyncio
import numpy as np

from io import BytesIO
from PIL import Image
from pyngrok import ngrok
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import StreamingResponse
from ultralytics import YOLO

# FastAPI 앱 생성
app = FastAPI()

# 2주차에서 학습한 최종 모델 로드 (exp3: epochs=30 + perspective/shear/degrees augmentation)
model = YOLO('runs/detect/train_results/exp3_augment/weights/best.pt')

# ngrok 인증 토큰 설정 (본인 ngrok 대시보드에서 발급받은 토큰으로 교체)
authtoken = "본인_authtoken_붙여넣기"
ngrok.set_auth_token(authtoken)

### fastapi 실행 함수 구현
- Input : 이미지 경로
- Output : 자유롭게 아이디어를 내서 결과를 출력시켜주세요.
  - 차량 정체 문제를 파악하기 위한 값으로 출력해주세요.
  - 예를 들어, 차량 개수 파악 / 차량 전체 면적 또는 비율 등 다양한 아이디어를 생각해서 구현해주세요.

In [ ]:
@app.post('/detect')
async def detect_api(file: UploadFile = File(...)):

    # 비동기적으로 파일 읽기
    image_bytes = await file.read()

    try:
        # RGB형태로 이미지 열기
        img = Image.open(BytesIO(image_bytes)).convert('RGB')

        # YOLO26 모델 추론 (학습 해상도와 동일하게 imgsz=224 사용)
        result = model.predict(source=img, conf=0.45, imgsz=224, verbose=False)[0]

        vehicle_cnt = len(result.boxes)
        img_w, img_h = img.size
        boxes_xyxy = result.boxes.xyxy.cpu().numpy()

        # 1) 면적 비율: 탐지된 차량 박스 면적 합 / 전체 이미지 면적 (%)
        box_areas = (boxes_xyxy[:, 2] - boxes_xyxy[:, 0]) * (boxes_xyxy[:, 3] - boxes_xyxy[:, 1])
        area_ratio = float(box_areas.sum() / (img_w * img_h) * 100) if vehicle_cnt > 0 else 0.0

        # 2) 차량 간 평균 간격(px): 차량이 2대 이상일 때, 각 차량과 가장 가까운 다른 차량 사이 거리의 평균
        if vehicle_cnt >= 2:
            centers = np.stack([
                (boxes_xyxy[:, 0] + boxes_xyxy[:, 2]) / 2,
                (boxes_xyxy[:, 1] + boxes_xyxy[:, 3]) / 2,
            ], axis=1)
            nearest_dists = []
            for i in range(len(centers)):
                other_centers = np.delete(centers, i, axis=0)
                nearest_dists.append(np.min(np.linalg.norm(other_centers - centers[i], axis=1)))
            avg_gap_px = float(np.mean(nearest_dists))
        else:
            avg_gap_px = None

        # 3) 정체 등급: 면적 비율 기준
        # 임계값(2% / 8.5%)은 삼정고가교(BC1000801) CCTV 라벨 데이터(AI-Hub 교통안전(Bbox), 750장) 기준
        # area_ratio 분포(중앙값 1.97%, 90분위 5.78%, 최댓값 19.89%)와 경계 구간 육안 확인으로 도출:
        #   ~7%는 육안상 혼잡 아님, ~12~14%는 육안상 혼잡으로 확인되어 그 사이인 8.5%를 congested 경계로 설정
        # HTTP 헤더에는 ASCII만 들어갈 수 있어서 한글 대신 영문 값을 사용
        if area_ratio < 2:
            congestion_level = 'smooth'
        elif area_ratio < 8.5:
            congestion_level = 'moderate'
        else:
            congestion_level = 'congested'

        # 결과 이미지: 박스가 그려진 이미지
        plotted_bgr = result.plot()
        plotted_rgb = Image.fromarray(plotted_bgr[:, :, ::-1])

        buf = io.BytesIO()
        plotted_rgb.save(buf, format='PNG')
        buf.seek(0)

        return StreamingResponse(
            buf,
            media_type='image/png',
            headers={
                'X-Vehicle-Count': str(vehicle_cnt),
                'X-Area-Ratio-Percent': f'{area_ratio:.2f}',
                'X-Avg-Gap-Px': f'{avg_gap_px:.1f}' if avg_gap_px is not None else 'NA',
                'X-Congestion-Level': congestion_level,
            }
        )

    except Exception as e:
        return {"error": "이미지 처리 실패", "details": str(e)}

# ngrok 터널 설정
port = 8000
public_url = ngrok.connect(port)
print(f"Public URL: {public_url}")

# FastAPI 서버 실행 (nest_asyncio 대신 Colab의 top-level await 사용 -> asyncio.run() 충돌 방지)
config = uvicorn.Config(app, host="0.0.0.0", port=port)
server = uvicorn.Server(config)
await server.serve()


### 지표 해석 시 유의사항 (Proxy 지표)

- **area_ratio** (탐지된 차량 박스 면적 합 / 전체 이미지 면적): 실제 정확한 도로 점유율이 아니라, 사진들 간의 **상대적인 정체 경향**을 나타내는 보조 지표입니다. 차량 박스끼리 겹치는 경우 겹친 영역이 중복 계산되어 값이 과대평가될 수 있고, 버스·트럭 등 대형차가 섞이면 차량 수가 적어도 면적 비율이 크게 올라가는 왜곡이 생길 수 있습니다. (예: `20201009_060005-090015_S_87615.jpg`는 차량 6대뿐인데 대형차 포함으로 area_ratio 10.49%까지 올라감)
- **avg_gap_px** (차량 간 평균 최근접 거리, px): 실제 정확한 차량 간 거리(m)가 아니라, 카메라 원근·해상도에 따라 달라지는 **상대적인 간격 경향**을 나타내는 보조 지표입니다. 같은 실제 거리라도 카메라와의 거리·각도에 따라 픽셀 값이 다르게 나올 수 있습니다.

→ 두 지표 모두 절대적인 물리량이 아니라 **정체 정도를 가늠하는 proxy 지표**로 활용하는 것이 적절합니다.

### 정체 등급 임계값(threshold) 설정 근거

임계값(smooth < 2%, moderate < 8.5%, congested ≥ 8.5%)은 임의로 정한 값이 아니라, AI-Hub "교통안전(Bbox)" 데이터셋 중 삼정고가교(BC1000801) CCTV 이미지 750장(실제 라벨링된 차량 bbox 포함)을 대상으로 area_ratio를 계산해 그 분포와 육안 확인을 근거로 정했습니다.

**1) 분포 확인**
- area_ratio 범위: 0.00% ~ 19.89%, 중앙값 1.97%, 90분위 5.78%
- 기존 기준(5% / 15%)으로는 congested가 750장 중 1장(0.1%)밖에 나오지 않아 사실상 쓰이지 않는 등급이었음

**2) 경계 구간 육안 확인**
| area_ratio | 파일명 | 육안 판단 |
|---|---|---|
| 1.97% | `20201007_170000-200027_S_158536.jpg` | 한산 |
| 6.98% | `20201007_170000-200027_S_162410.jpg` | 혼잡 아님 |
| 12.06% / 13.14% | `20201008_055955-090009_S_208494.jpg` 등 | 혼잡 |
| 19.89% | `20201007_170000-200027_S_151086.jpg` | 완전 정체 |

→ 6.98%(혼잡 아님)와 12~13%(혼잡) 사이에 실제 경계가 있다고 판단하여 moderate/congested 경계를 **8.5%**로 설정

**3) 새 기준으로 재검증 (GT bbox 기준)**
| 기준 | smooth | moderate | congested |
|---|---|---|---|
| 기존 5% / 15% | 85.9% | 14.0% | 0.1% |
| 신규 2% / 8.5% | 50.8% | 46.3% | 2.9% |

**4) ⚠ 실제 배포 모델(best.pt)로 재검증 — 중요 발견**

위 threshold는 모두 **정답(GT) bbox** 기준으로 계산한 것입니다. 그런데 실제 `/detect` API는 `best.pt`가 **직접 탐지한** bbox로 area_ratio를 계산하므로, 실제 서비스에 그대로 적용되는지 확인이 필요했습니다.

`20201007_170000-200027_S_162410.jpg`(야간, GT area_ratio 6.98%, 14대)와 `20201007_120000-150012_S_254826.jpg`(주간, GT area_ratio 7.19%, 7대) 두 장을 실제 `/detect` API에 넣어본 결과:

| 파일 | 시간대 | GT area_ratio | GT 차량 수 | API 응답 vehicle_count | API 응답 congestion_level |
|---|---|---|---|---|---|
| `..._S_162410.jpg` | 야간 | 6.98% | 14대 | **0** | smooth |
| `..._S_254826.jpg` | 주간 | 7.19% | 7대 | **0** | smooth |

낮/밤 두 경우 모두 실제로는 차량이 여러 대 있음에도 탐지 결과가 0대였습니다. 조명(야간) 조건과 무관하게 동일하게 실패했으므로, 2주차에 확인한 **각도(시점) domain gap**이 원인이라고 판단됩니다 — 학습 데이터가 synthetic 수직 항공뷰 위주였는데, 삼정고가교 CCTV는 비스듬한 각도로 촬영되어 모델이 학습 때 보지 못한 시각 패턴이기 때문입니다.

**결론**: 위에서 산정한 threshold(2%/8.5%) 자체는 데이터 기반으로 타당하게 도출되었지만, **현재 배포된 모델의 탐지 성능 문제로 이 threshold가 이 카메라에서는 사실상 적용되지 못하고 있습니다**(항상 area_ratio 0%, smooth로 오판). threshold 설계보다 더 근본적인 문제이며, 실제 각도 이미지로 fine-tuning해야 이 threshold가 실질적으로 의미를 가집니다.

**한계**: 이 임계값은 삼정고가교 CCTV 1개 지점의 카메라 각도·높이 기준으로 도출된 것으로, 다른 카메라(각도·거리·차선 수가 다른 지점)에는 그대로 적용하기 어려울 수 있으며, 지점별로 재검증이 필요합니다.